9. In this exercise, we will predict the number of applications received using the other variables in the College data set.

In [1]:
from google.colab import files
uploaded = files.upload()

Saving College.csv to College.csv


In [2]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, KFold, GridSearchCV
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA

# -------------------------------------------------
# 讀資料與前處理
# -------------------------------------------------
# College.csv 通常第一欄是校名，當作 row index 即可
college = pd.read_csv("College.csv", index_col=0)

# 題目要預測的 Y: Apps (number of applications)
y = college["Apps"]

# 對類別變數 Private 做 one-hot
X = college.drop(columns=["Apps"])
X = pd.get_dummies(X, drop_first=True)   # 會把 Private 轉為 0/1

(a)

Split the data set into a training set and a test set.

In [3]:
# (a) 切訓練 / 測試集（這裡用 50% / 50%，random_state=1 對應書上 set.seed(1)）
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.5, random_state=1
)

print("Train size:", X_train.shape[0])
print("Test  size:", X_test.shape[0])

Train size: 388
Test  size: 389


(b)

Fit a linear model using least squares on the training set, and report the test error obtained.

In [4]:
# (b) 線性迴歸（最小平方法）+ 測試 MSE
lin_reg = LinearRegression()
lin_reg.fit(X_train, y_train)

y_pred_lin = lin_reg.predict(X_test)
mse_lin = mean_squared_error(y_test, y_pred_lin)

print("\n(b) Linear regression")
print("Test MSE:", mse_lin)


(b) Linear regression
Test MSE: 1425055.5873112094


(c)

Fit a ridge regression model on the training set, with λ chosen by cross-validation. Report the test error obtained.

In [5]:
# (c) Ridge regression（標準化 + Ridge），用 CV 選 alpha
ridge_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("ridge", Ridge())
])

# alpha (λ) 搜尋範圍，可依需要調整
alphas_ridge = np.logspace(-3, 5, 50)

param_grid_ridge = {
    "ridge__alpha": alphas_ridge
}

cv = KFold(n_splits=10, shuffle=True, random_state=1)

ridge_cv = GridSearchCV(
    ridge_pipe,
    param_grid_ridge,
    cv=cv,
    scoring="neg_mean_squared_error"
)
ridge_cv.fit(X_train, y_train)

ridge_best = ridge_cv.best_estimator_
best_alpha_ridge = ridge_cv.best_params_["ridge__alpha"]

y_pred_ridge = ridge_best.predict(X_test)
mse_ridge = mean_squared_error(y_test, y_pred_ridge)

print("\n(c) Ridge regression")
print("Best alpha (λ):", best_alpha_ridge)
print("Test MSE:", mse_ridge)


(c) Ridge regression
Best alpha (λ): 3.906939937054613
Test MSE: 1521487.2590996753


(d)

Fit a lasso model on the training set, with λ chosen by cross-validation. Report the test error obtained, along with the number of non-zero coefficient estimates.

In [6]:
# (d) Lasso（標準化 + Lasso），用 CV 選 alpha
# -------------------------------------------------
lasso_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("lasso", Lasso(max_iter=10000))
])

alphas_lasso = np.logspace(-3, 1, 50)

param_grid_lasso = {
    "lasso__alpha": alphas_lasso
}

lasso_cv = GridSearchCV(
    lasso_pipe,
    param_grid_lasso,
    cv=cv,
    scoring="neg_mean_squared_error"
)
lasso_cv.fit(X_train, y_train)

lasso_best = lasso_cv.best_estimator_
best_alpha_lasso = lasso_cv.best_params_["lasso__alpha"]

y_pred_lasso = lasso_best.predict(X_test)
mse_lasso = mean_squared_error(y_test, y_pred_lasso)

# 非零係數數量
lasso_coefs = lasso_best.named_steps["lasso"].coef_
num_nonzero = np.sum(lasso_coefs != 0)

print("\n(d) Lasso")
print("Best alpha (λ):", best_alpha_lasso)
print("Test MSE:", mse_lasso)
print("Number of non-zero coefficients:", num_nonzero)


(d) Lasso
Best alpha (λ): 6.866488450042998
Test MSE: 1409868.4418062235
Number of non-zero coefficients: 16


(e)

Fit a PCR model on the training set, with M chosen by cross-validation. Report the test error obtained, along with the value of M selected by cross-validation.

In [7]:
# (e) PCR：標準化 + PCA + 線性迴歸，用 CV 選 M（主成分數）
pcr_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("pca", PCA()),
    ("linreg", LinearRegression())
])

# 從 1 個主成分到全部主成分都試一次
n_features = X_train.shape[1]
param_grid_pcr = {
    "pca__n_components": list(range(1, n_features + 1))
}

pcr_cv = GridSearchCV(
    pcr_pipe,
    param_grid_pcr,
    cv=cv,
    scoring="neg_mean_squared_error"
)
pcr_cv.fit(X_train, y_train)

pcr_best = pcr_cv.best_estimator_
best_M = pcr_cv.best_params_["pca__n_components"]

y_pred_pcr = pcr_best.predict(X_test)
mse_pcr = mean_squared_error(y_test, y_pred_pcr)

print("\n(e) PCR (Principal Components Regression)")
print("Best number of components M:", best_M)
print("Test MSE:", mse_pcr)


(e) PCR (Principal Components Regression)
Best number of components M: 16
Test MSE: 1455919.8617456967
